In [1]:
#!pip install flask
#!pip install nest-asyncio
#!pip install flask dill numpy nest_asyncio scikit-learn

In [2]:
import numpy as np
import pandas as pd
from flask import Flask, request, render_template
import dill
import nest_asyncio
import traceback

In [3]:
try:
    with open("mdl.pkl", "rb") as file:
        model = dill.load(file)
    print("Model berhasil dimuat!")
    # Lakukan tes sederhana
    test_input = [[1, 2, 3, 4]]  # Sesuaikan dengan format input model Anda
    prediction = model.predict(test_input)
    print("Prediksi percobaan:", prediction)
except Exception as e:
    print(f"Gagal memuat model: {e}")
    import traceback
    print(traceback.format_exc())

Model berhasil dimuat!
Prediksi percobaan: [0]


C:\Users\Rebecca YBS\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [4]:
# Terapkan nest_asyncio
nest_asyncio.apply()

# Buat aplikasi Flask
app = Flask(__name__)

# Global variables untuk model dan dataset
model = None
df = None

# Fungsi untuk memuat model
def load_model():
    global model
    try:
        with open("mdl.pkl", "rb") as file:
            model = dill.load(file)
        print("Model berhasil dimuat!")
    except FileNotFoundError:
        print("Model file 'mdl.pkl' tidak ditemukan. Pastikan file ada di direktori yang benar.")
        raise
    except Exception as e:
        print(f"Gagal memuat model: {e}")
        print(traceback.format_exc())
        raise

# Fungsi untuk memuat dataset
def load_dataset():
    global df
    try:
        df = pd.read_csv("../Dataset/Kontekstual DM/final_processed_data.csv")
        print("Dataset berhasil dimuat!")
    except FileNotFoundError:
        print("Dataset file tidak ditemukan. Pastikan file ada di direktori yang benar.")
        raise
    except Exception as e:
        print(f"Gagal memuat dataset: {e}")
        print(traceback.format_exc())
        raise

In [5]:
# Mapping kelas status kunjungan
status_kunjungan_mapping = {
    0: 'Berobat jalan',
    1: 'Rujuk lanjut', 
    2: 'Sembuh',
    3: 'Kunjungan sehat',
    4: 'Lain-lain/Pulang paksa/Meninggal'
}

In [6]:
# Route untuk halaman utama
@app.route("/")
def home():
    # Ambil data untuk dropdown dari dataset
    provinsi_list = df['Nama_Provinsi'].unique()
    poliklinik_list = df['Jenis_Poliklinik'].unique()
    diagnosis_list = df['Nama_Diagnosis'].unique()
    jenis_kunjungan_list = df['Jenis_Kunjungan'].unique()

    # Kirim data ke template
    return render_template("index.html", 
                           provinsi_list=provinsi_list,
                           poliklinik_list=poliklinik_list,
                           diagnosis_list=diagnosis_list,
                           jenis_kunjungan_list=jenis_kunjungan_list)

In [7]:
# Route untuk prediksi
@app.route("/predict", methods=["POST"])
def predict():
    if model is None:
        return render_template("index.html", prediction_text="Model belum dimuat. Hubungi administrator.")

    try:
        # Ambil input dari user
        nama_provinsi = request.form.get('Nama_Provinsi')
        jenis_poliklinik = request.form.get('Jenis_Poliklinik')
        nama_diagnosis = request.form.get('Nama_Diagnosis')
        jenis_kunjungan = request.form.get('Jenis_Kunjungan')

        # Cek jika ada input yang kosong
        if not nama_provinsi or not jenis_poliklinik or not nama_diagnosis or not jenis_kunjungan:
            # Ambil data untuk dropdown agar tidak kosong
            provinsi_list = df['Nama_Provinsi'].unique()
            poliklinik_list = df['Jenis_Poliklinik'].unique()
            diagnosis_list = df['Nama_Diagnosis'].unique()
            jenis_kunjungan_list = df['Jenis_Kunjungan'].unique()

            return render_template("index.html", prediction_text="Semua kolom harus diisi!",
                                   Nama_Provinsi=nama_provinsi, Jenis_Poliklinik=jenis_poliklinik,
                                   Nama_Diagnosis=nama_diagnosis, Jenis_Kunjungan=jenis_kunjungan,
                                   provinsi_list=provinsi_list,
                                   poliklinik_list=poliklinik_list,
                                   diagnosis_list=diagnosis_list,
                                   jenis_kunjungan_list=jenis_kunjungan_list)

        # Cek dan konversi nama ke kode menggunakan dataset
        try:
            kode_provinsi = df.loc[df['Nama_Provinsi'] == nama_provinsi, 'Kode_Provinsi'].values[0]
            kode_poliklinik = df.loc[df['Jenis_Poliklinik'] == jenis_poliklinik, 'Kode_Jenis_Poliklinik'].values[0]
            kode_diagnosis = df.loc[df['Nama_Diagnosis'] == nama_diagnosis, 'Kode_Diagnosis'].values[0]
            kode_jenis_kunjungan = df.loc[df['Jenis_Kunjungan'] == jenis_kunjungan, 'Kode_Jenis_Kunjungan'].values[0]
        except IndexError:
            # Ambil data untuk dropdown agar tidak kosong
            provinsi_list = df['Nama_Provinsi'].unique()
            poliklinik_list = df['Jenis_Poliklinik'].unique()
            diagnosis_list = df['Nama_Diagnosis'].unique()
            jenis_kunjungan_list = df['Jenis_Kunjungan'].unique()

            return render_template("index.html", prediction_text="Data yang dimasukkan tidak ditemukan di dataset.",
                                   Nama_Provinsi=nama_provinsi, Jenis_Poliklinik=jenis_poliklinik,
                                   Nama_Diagnosis=nama_diagnosis, Jenis_Kunjungan=jenis_kunjungan,
                                   provinsi_list=provinsi_list,
                                   poliklinik_list=poliklinik_list,
                                   diagnosis_list=diagnosis_list,
                                   jenis_kunjungan_list=jenis_kunjungan_list)

        # Pastikan semua kode ditemukan
        input_features = [kode_provinsi, kode_poliklinik, kode_diagnosis, kode_jenis_kunjungan]

        # Konversi ke array numpy
        features = np.array([input_features])

        # Prediksi probabilitas
        probabilities = model.predict_proba(features)

        # Hitung jumlah pasien untuk setiap kelas (dalam bentuk prosentase)
        jumlah_pasien = {status_kunjungan_mapping[i]: round(probabilities[0][i] * 100) for i in range(len(probabilities[0]))}

        # Menyiapkan output prediksi dalam bentuk tabel
        return render_template("index.html", prediction_results=jumlah_pasien,
                               Nama_Provinsi=nama_provinsi, Jenis_Poliklinik=jenis_poliklinik,
                               Nama_Diagnosis=nama_diagnosis, Jenis_Kunjungan=jenis_kunjungan,
                               provinsi_list=df['Nama_Provinsi'].unique(),
                               poliklinik_list=df['Jenis_Poliklinik'].unique(),
                               diagnosis_list=df['Nama_Diagnosis'].unique(),
                               jenis_kunjungan_list=df['Jenis_Kunjungan'].unique())

    except Exception as e:
        return render_template("index.html", prediction_text=f"Terjadi kesalahan: {str(e)}")

In [ ]:
# Muat model dan dataset sebelum menjalankan server
load_model()
load_dataset()

# Jalankan aplikasi
if __name__ == "__main__":
    try:
        app.run(debug=True, host='0.0.0.0', port=5000, use_reloader=False)
    except Exception as e:
        print(f"Gagal menjalankan server: {e}")
        print(traceback.format_exc())

Model berhasil dimuat!
Dataset berhasil dimuat!
 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [12/Dec/2024 15:11:27] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [12/Dec/2024 15:11:27] "GET /favicon.ico HTTP/1.1" 404 -
C:\Users\Rebecca YBS\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
127.0.0.1 - - [12/Dec/2024 15:11:46] "POST /predict HTTP/1.1" 200 -
C:\Users\Rebecca YBS\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
127.0.0.1 - - [12/Dec/2024 15:12:13] "POST /predict HTTP/1.1" 200 -
C:\Users\Rebecca YBS\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
127.0.0.1 - - [12/Dec/2024 15:12:58] "POST /